# 03 — RFM Customer Segmentation

Score every unique customer on **Recency / Frequency / Monetary** value, assign a business segment label, and produce visualisations and a rollup summary ready for Power BI.

| Step | Input | Output |
|------|-------|--------|
| 0. Setup | PostgreSQL star schema | `fact_df` (merged DataFrame) |
| 1. Compute RFM | `fact_df` | `rfm` (recency, frequency, monetary) |
| 2. Score & Segment | `rfm` | `rfm` + R/F/M scores + segment label |
| 3. Visualise | `rfm` + `summary` | Charts |
| 4. Business Interpretation | `summary` | Narrative + actions |
| 5. Export | `rfm` | `customer_rfm.csv` + `dim_customer_rfm` table |

## 0. Setup

In [ ]:
from __future__ import annotations

import logging
import sys
import warnings

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
sys.path.append('..')

from src.db import get_engine, load_df_to_sql
from src.config import PROCESSED_DATA_DIR
from src.rfm import compute_rfm, score_rfm, assign_segment, segment_summary

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted")

engine = get_engine()
print("Connected to:", engine.url)

In [ ]:
# Join fact_order_item with dim_customer to obtain customer_unique_id.
# Group to order grain here so payment_value (an order-level total) is
# not double-counted when there are multiple line items per order.
SQL = """
    SELECT
        dc.customer_unique_id,
        f.order_id,
        MAX(f.date_id)        AS date_id,
        MAX(f.payment_value)  AS payment_value
    FROM fact_order_item f
    JOIN dim_customer dc ON f.customer_id = dc.customer_id
    WHERE f.order_status = 'delivered'
    GROUP BY dc.customer_unique_id, f.order_id
"""

fact_df = pd.read_sql(SQL, engine, parse_dates=["date_id"])
print(f"Loaded {len(fact_df):,} order rows")
print(f"Unique customers : {fact_df['customer_unique_id'].nunique():,}")
print(f"Date range       : {fact_df['date_id'].min().date()} → {fact_df['date_id'].max().date()}")
fact_df.head()

## 1. Compute RFM

In [ ]:
rfm = compute_rfm(fact_df)
print(f"RFM table: {len(rfm):,} unique customers")
rfm.describe().round(2)

In [ ]:
# Frequency distribution — most Olist customers are one-time buyers
print("Frequency distribution (top 10 values):")
print(rfm["frequency"].value_counts().head(10).to_string())
print(f"\nCustomers with frequency == 1: {(rfm['frequency'] == 1).mean():.1%}")

## 2. Score & Segment

In [ ]:
rfm = score_rfm(rfm)                                   # adds R_score, F_score, M_score
rfm["segment"] = rfm.apply(assign_segment, axis=1)    # maps (R,F) → label

print("Score distributions:")
for col in ["R_score", "F_score", "M_score"]:
    print(f"  {col}: {rfm[col].value_counts().sort_index().to_dict()}")

rfm.head()

In [ ]:
print("Segment customer counts:")
print(rfm["segment"].value_counts().to_string())

## 3. Visualise

In [ ]:
# 3a — RFM metric distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rfm["recency"], bins=50, color="steelblue", edgecolor="white", linewidth=0.4)
axes[0].set_title("Recency (days since last order)", fontsize=11)
axes[0].set_xlabel("Days")
axes[0].set_ylabel("Customers")

axes[1].hist(rfm["frequency"].clip(upper=10), bins=10, color="seagreen", edgecolor="white", linewidth=0.4)
axes[1].set_title("Frequency (orders, capped at 10)", fontsize=11)
axes[1].set_xlabel("Orders")

axes[2].hist(rfm["monetary"].clip(upper=rfm["monetary"].quantile(0.99)), bins=50,
             color="coral", edgecolor="white", linewidth=0.4)
axes[2].set_title("Monetary (BRL, clipped at p99)", fontsize=11)
axes[2].set_xlabel("BRL")

fig.suptitle("RFM Metric Distributions", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 3b — Customer count by segment (sorted descending)
# pandas 2.x: value_counts().reset_index() yields columns ['segment', 'count']
seg_counts = (
    rfm["segment"].value_counts()
    .reset_index()
    .sort_values("count", ascending=True)   # ascending for horizontal bar
)

palette = sns.color_palette("tab10", len(seg_counts))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(seg_counts["segment"], seg_counts["count"], color=palette)
ax.bar_label(bars, fmt="%d", padding=4, fontsize=9)
ax.set_title("Customer Count by RFM Segment", fontsize=13, fontweight="bold")
ax.set_xlabel("Customers")
ax.set_xlim(right=seg_counts["count"].max() * 1.12)
plt.tight_layout()
plt.show()

In [ ]:
# 3c — Revenue contribution by segment (% of total)
summary = segment_summary(rfm)

plot_data = summary.sort_values("pct_revenue", ascending=True)
palette2 = sns.color_palette("tab10", len(plot_data))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(plot_data["segment"], plot_data["pct_revenue"], color=palette2)
ax.bar_label(bars, fmt="%.1f%%", padding=4, fontsize=9)
ax.set_title("Revenue Contribution by Segment (% of Total)", fontsize=13, fontweight="bold")
ax.set_xlabel("% of Total Revenue")
ax.set_xlim(right=plot_data["pct_revenue"].max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# 3d — Recency vs Monetary scatter coloured by segment (Plotly for interactivity)
fig = px.scatter(
    rfm,
    x="recency",
    y="monetary",
    color="segment",
    size="frequency",
    size_max=12,
    opacity=0.55,
    title="Recency vs Monetary Value by Segment",
    labels={
        "recency": "Recency (days since last order)",
        "monetary": "Monetary (total spend, BRL)",
        "frequency": "Orders",
        "segment": "Segment",
    },
    hover_data=["frequency", "R_score", "F_score", "M_score"],
    height=520,
)
fig.update_layout(legend_title_text="Segment", font_size=12)
fig.show()

In [ ]:
# 3e — Heatmap: average monetary value by R_score × F_score
pivot = (
    rfm.groupby(["R_score", "F_score"])["monetary"]
    .mean()
    .reset_index()
    .pivot_table(index="R_score", columns="F_score", values="monetary", fill_value=0)
    .sort_index(ascending=False)   # R=5 at top
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".0f",
    cmap="YlOrRd",
    linewidths=0.5,
    ax=ax,
    cbar_kws={"label": "Avg Monetary (BRL)"},
)
ax.set_title("Average Monetary Value by R_score × F_score", fontsize=13, fontweight="bold")
ax.set_xlabel("F_score (Frequency)")
ax.set_ylabel("R_score (Recency)")
plt.tight_layout()
plt.show()

## 4. Business Interpretation

In [ ]:
SEGMENT_ACTIONS: dict[str, str] = {
    "Champions":          "protect via VIP program, early access to new products, loyalty rewards",
    "Loyal":              "upsell premium products, offer referral incentives",
    "Potential Loyalist": "nurture with personalised offers to increase purchase frequency",
    "At Risk":            "win-back campaign — targeted discount + reactivation email (priority)",
    "Can't Lose Them":    "urgent personal outreach, exclusive high-value retention offer",
    "Hibernating":        "low-cost 'We miss you' email with small incentive",
    "Lost":               "minimal spend — broad reactivation only if cost-effective",
    "Others":             "investigate purchasing pattern and reclassify",
}

lines = []
for _, row in summary.iterrows():
    seg = row["segment"]
    action = SEGMENT_ACTIONS.get(seg, "monitor")
    lines.append(
        f"**{seg}**: {int(row['count']):,} customers "
        f"({row['pct_customers']}% of base) · "
        f"${row['total_monetary']:,.0f} revenue "
        f"({row['pct_revenue']}% of total) · "
        f"avg {row['avg_recency']:.0f} days since last order — "
        f"*{action}*"
    )

display(Markdown("\n\n".join(lines)))

## 5. Export

In [ ]:
# Save to CSV for downstream tools (Excel, Power BI flat-file connector)
csv_path = PROCESSED_DATA_DIR / "customer_rfm.csv"
rfm.to_csv(csv_path, index=False)
print(f"CSV saved: {csv_path}  ({len(rfm):,} rows)")

# Load to PostgreSQL so Power BI can join on customer_unique_id
rows_written = load_df_to_sql(rfm, "dim_customer_rfm", if_exists="replace")
print(f"PostgreSQL table 'dim_customer_rfm' loaded: {rows_written:,} rows")

In [ ]:
# Verify round-trip
check = pd.read_sql("SELECT COUNT(*) AS n FROM dim_customer_rfm", engine)
assert check.iloc[0]["n"] == len(rfm), "Row count mismatch after load!"
print(f"Round-trip verified: {check.iloc[0]['n']:,} rows in dim_customer_rfm")

## 6. Resume Bullet Draft

*Update the numbers below by running the cell after this one, then copy into your CV.*

In [ ]:
n_customers   = len(rfm)
n_segments    = rfm["segment"].nunique()
total_rev     = rfm["monetary"].sum()

at_risk_row   = summary[summary["segment"].isin(["At Risk", "Can't Lose Them"])]
at_risk_n     = int(at_risk_row["count"].sum())
at_risk_rev   = at_risk_row["total_monetary"].sum()

champ_row     = summary[summary["segment"] == "Champions"]
champ_n       = int(champ_row["count"].iloc[0]) if len(champ_row) else 0
champ_pct_rev = float(champ_row["pct_revenue"].iloc[0]) if len(champ_row) else 0.0

bullets = f"""
**Resume Bullet Draft** — update metrics from the numbers below, then trim to ≤ 2 lines each.

- Engineered end-to-end RFM segmentation pipeline in Python (pandas + SQLAlchemy) on {n_customers:,} \
Olist e-commerce customers, classifying them into {n_segments} behavioural segments (Champions, Loyal, \
At Risk, etc.) stored in a PostgreSQL star schema and visualised in Power BI, reducing ad-hoc \
reporting time from hours to minutes.

- Identified {at_risk_n:,} at-risk / can't-lose customers representing ${at_risk_rev:,.0f} in lifetime \
revenue ({at_risk_rev / total_rev:.1%} of total); proposed automated win-back email sequences with \
estimated 15–20% reactivation rate based on industry benchmarks.

- Champions segment ({champ_n:,} customers) drives {champ_pct_rev:.1f}% of total revenue — built \
VIP retention playbook (early-access launches, loyalty rewards) to protect this cohort and reduce \
revenue concentration risk.
"""

display(Markdown(bullets))